In [0]:
use catalog identifier(:catalog);
use schema identifier(:schema);

In [0]:
create or replace table dim_compute (
  compute_key bigint generated by default as identity (start with 1 increment by 1) comment 'Surrogate primary key (PK) of the table.',
  warehouse_id string comment 'The natural key of the warehouse from system.compute.warehouses.',
  workspace_id string comment 'The ID of the workspace where the warehouse is deployed.',
  account_id string comment 'The ID of the Databricks account.',
  warehouse_name string comment 'The name of the SQL warehouse.',
  warehouse_type string comment 'The type of SQL warehouse. Possible values are CLASSIC, PRO, and SERVERLESS.',
  warehouse_channel string comment 'The channel of the SQL warehouse. Possible values are CURRENT and PREVIEW.',
  warehouse_size string comment 'The cluster size of the SQL warehouse. Possible values are 2X_SMALL, X_SMALL, SMALL, MEDIUM, LARGE, X_LARGE, 2X_LARGE, 3X_LARGE, and 4X_LARGE.',
  min_clusters int comment 'The minimum number of clusters permitted.',
  max_clusters int comment 'The maximum number of clusters permitted.',
  auto_stop_minutes int comment 'The number of minutes before the SQL warehouse auto-stops due to inactivity.',
  change_time timestamp comment 'Timestamp of change to the SQL warehouse definition.',
  delete_time timestamp comment 'Timestamp of when the SQL warehouse was deleted. The value is null if the SQL warehouse is not deleted.',
  compute_type string not null comment 'The type of compute (Warehouse, Serverless, etc.)',
  effective_start timestamp comment 'Timestamp when this version of the record became effective.',
  effective_end timestamp comment 'Timestamp when this version was superseded. NULL indicates the current active record.',
  is_current boolean comment 'Flag indicating whether this is the current active version of the record.',
  primary key (compute_key) rely
)
comment 'DBSQL warehouse dimension (SCD Type 2) sourced from system.compute.warehouses'
cluster by (compute_key);

In [0]:
insert overwrite dim_compute (
  warehouse_id,
  workspace_id,
  account_id,
  warehouse_name,
  warehouse_type,
  warehouse_channel,
  warehouse_size,
  min_clusters,
  max_clusters,
  auto_stop_minutes,
  change_time,
  delete_time,
  compute_type,
  effective_start,
  effective_end,
  is_current
)
-- Warehouses SCD Type 2: retain full history of configuration changes
select
  warehouse_id,
  workspace_id,
  account_id,
  warehouse_name,
  warehouse_type,
  warehouse_channel,
  warehouse_size,
  min_clusters,
  max_clusters,
  auto_stop_minutes,
  change_time,
  delete_time,
  'Warehouse' as compute_type,
  change_time as effective_start,
  lead(change_time) over (partition by warehouse_id order by change_time) as effective_end,
  row_number() over (partition by warehouse_id order by change_time desc) = 1 as is_current
from system.compute.warehouses;

In [0]:
-- unknown row (compute_key = -1)
insert into dim_compute (
  compute_key,
  warehouse_id,
  workspace_id,
  account_id,
  warehouse_name,
  warehouse_type,
  warehouse_channel,
  warehouse_size,
  min_clusters,
  max_clusters,
  auto_stop_minutes,
  change_time,
  delete_time,
  compute_type,
  effective_start,
  effective_end,
  is_current
)
values (
  -1,
  'unknown',
  'unknown',
  'unknown',
  'unknown',
  'unknown',
  'unknown',
  'unknown',
  null,
  null,
  null,
  null,
  null,
  'unknown',
  current_timestamp(),
  null,
  true
);

In [0]:
optimize dim_compute

In [0]:
analyze table dim_compute compute statistics for all columns;

In [0]:
vacuum dim_compute